In [4]:
# Instalasi library wajib untuk Local LLM
!pip install -q transformers accelerate bitsandbytes pandas tqdm

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from tqdm import tqdm
from google.colab import drive
import re

# Hubungkan ke Drive
drive.mount('/content/drive')

print("\n⚙️ Memuat Qwen 2.5 7B Instruct ke GPU (Menggunakan kompresi 4-bit)...")
# Setup kompresi agar muat di RAM 16GB Colab
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model_id = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)
print("✅ Qwen berhasil bersarang di GPU Colab Anda!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.8 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

⚙️ Memuat Qwen 2.5 7B Instruct ke GPU (Menggunakan kompresi 4-bit)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

✅ Qwen berhasil bersarang di GPU Colab Anda!


In [ ]:
# GANTI INI sesuai lokasi file INET Anda di Drive
path_input = "/content/drive/MyDrive/NLP_qwen/stockbit_INET_stream.csv"
path_output = "/content/drive/MyDrive/NLP_qwen/INET_labeled.csv"

df = pd.read_csv(path_input)

# Fungsi super pintar untuk memaksa Qwen menjawab 1 kata saja dengan Few-Shot Prompting
def tebak_sentimen(teks_komentar):
    # Prompt sistem dengan contoh agar Qwen langsung jago bahasa Stockbit
    messages = [
        {"role": "system", "content": """Kamu adalah ahli sentimen saham ritel Indonesia.
Tugasmu hanya menjawab dengan SATU KATA: 'Positif', 'Negatif', atau 'Netral'.
Contoh:
- 'Serok bawah mumpung diskon' -> Positif
- 'Aduh nyangkut di pucuk, CL aja deh' -> Negatif
- 'Hold dulu, nunggu laporan keuangan' -> Netral
- 'Bandar guyur terus' -> Negatif
- 'ARA besok nih!' -> Positif"""},
        {"role": "user", "content": f"Tentukan sentimen kalimat ini: '{teks_komentar}'"}
    ]

    text_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text_prompt], return_tensors="pt").to(model.device)

    # Generate jawaban maksimal 10 token biar cepat
    outputs = model.generate(**inputs, max_new_tokens=10, temperature=0.1, do_sample=False)

    # Ambil teks jawaban saja (buang prompt-nya)
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

    # Bersihkan jawaban (jaga-jaga kalau Qwen ngomong panjang lebar)
    hasil = response.strip().lower()
    if 'positif' in hasil: return 'Positif'
    elif 'negatif' in hasil: return 'Negatif'
    elif 'netral' in hasil: return 'Netral'
    else: return 'Netral' # Default kalau jawabannya aneh

print(f"Memulai pelabelan {len(df)} baris data INET...")
labels = []

# Proses satu per satu dengan progress bar
for text in tqdm(df['content'].astype(str), desc="Melabeli"):
    labels.append(tebak_sentimen(text))

# Simpan hasil ke kolom baru
df['label'] = labels

# Simpan ke CSV baru
df.to_csv(path_output, index=False)
print(f"\n🎉 SELESAI! Data berlabel disimpan di: {path_output}")
print(df['label'].value_counts())

Memulai pelabelan 1000 baris data INET...


Melabeli: 100%|██████████| 1000/1000 [27:38<00:00,  1.66s/it]


🎉 SELESAI! Data berlabel disimpan di: /content/drive/MyDrive/NLP_qwen/INET_labeled.csv
label
Positif    419
Negatif    339
Netral     242
Name: count, dtype: int64


In [ ]:
path_input = "/content/drive/MyDrive/NLP_qwen/stockbit_DEWA_stream.csv"
path_output = "/content/drive/MyDrive/NLP_qwen/DEWA_labeled.csv"

df = pd.read_csv(path_input)

# Fungsi super pintar untuk memaksa Qwen menjawab 1 kata saja dengan Few-Shot Prompting
def tebak_sentimen(teks_komentar):
    # Prompt sistem dengan contoh agar Qwen langsung jago bahasa Stockbit
    messages = [
        {"role": "system", "content": """Kamu adalah ahli sentimen saham ritel Indonesia.
Tugasmu hanya menjawab dengan SATU KATA: 'Positif', 'Negatif', atau 'Netral'.
Contoh:
- 'Serok bawah mumpung diskon' -> Positif
- 'Aduh nyangkut di pucuk, CL aja deh' -> Negatif
- 'Hold dulu, nunggu laporan keuangan' -> Netral
- 'Bandar guyur terus' -> Negatif
- 'ARA besok nih!' -> Positif"""},
        {"role": "user", "content": f"Tentukan sentimen kalimat ini: '{teks_komentar}'"}
    ]

    text_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text_prompt], return_tensors="pt").to(model.device)

    # Generate jawaban maksimal 10 token biar cepat
    outputs = model.generate(**inputs, max_new_tokens=10, temperature=0.1, do_sample=False)

    # Ambil teks jawaban saja (buang prompt-nya)
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

    # Bersihkan jawaban (jaga-jaga kalau Qwen ngomong panjang lebar)
    hasil = response.strip().lower()
    if 'positif' in hasil: return 'Positif'
    elif 'negatif' in hasil: return 'Negatif'
    elif 'netral' in hasil: return 'Netral'
    else: return 'Netral' # Default kalau jawabannya aneh

print(f"Memulai pelabelan {len(df)} baris data INET...")
labels = []

# Proses satu per satu dengan progress bar
for text in tqdm(df['content'].astype(str), desc="Melabeli"):
    labels.append(tebak_sentimen(text))

# Simpan hasil ke kolom baru
df['label'] = labels

# Simpan ke CSV baru
df.to_csv(path_output, index=False)
print(f"\n🎉 SELESAI! Data berlabel disimpan di: {path_output}")
print(df['label'].value_counts())

Memulai pelabelan 1000 baris data INET...


Melabeli: 100%|██████████| 1000/1000 [29:21<00:00,  1.76s/it]


🎉 SELESAI! Data berlabel disimpan di: /content/drive/MyDrive/NLP_qwen/DEWA_labeled.csv
label
Positif    468
Negatif    291
Netral     241
Name: count, dtype: int64


In [ ]:
path_input = "/content/drive/MyDrive/NLP_qwen/stockbit_AADI_stream.csv"
path_output = "/content/drive/MyDrive/NLP_qwen/AADI_labeled.csv"

df = pd.read_csv(path_input)

# Fungsi super pintar untuk memaksa Qwen menjawab 1 kata saja dengan Few-Shot Prompting
def tebak_sentimen(teks_komentar):
    # Prompt sistem dengan contoh agar Qwen langsung jago bahasa Stockbit
    messages = [
        {"role": "system", "content": """Kamu adalah ahli sentimen saham ritel Indonesia.
Tugasmu hanya menjawab dengan SATU KATA: 'Positif', 'Negatif', atau 'Netral'.
Contoh:
- 'Serok bawah mumpung diskon' -> Positif
- 'Aduh nyangkut di pucuk, CL aja deh' -> Negatif
- 'Hold dulu, nunggu laporan keuangan' -> Netral
- 'Bandar guyur terus' -> Negatif
- 'ARA besok nih!' -> Positif"""},
        {"role": "user", "content": f"Tentukan sentimen kalimat ini: '{teks_komentar}'"}
    ]

    text_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text_prompt], return_tensors="pt").to(model.device)

    # Generate jawaban maksimal 10 token biar cepat
    outputs = model.generate(**inputs, max_new_tokens=10, temperature=0.1, do_sample=False)

    # Ambil teks jawaban saja (buang prompt-nya)
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

    # Bersihkan jawaban (jaga-jaga kalau Qwen ngomong panjang lebar)
    hasil = response.strip().lower()
    if 'positif' in hasil: return 'Positif'
    elif 'negatif' in hasil: return 'Negatif'
    elif 'netral' in hasil: return 'Netral'
    else: return 'Netral' # Default kalau jawabannya aneh

print(f"Memulai pelabelan {len(df)} baris data INET...")
labels = []

# Proses satu per satu dengan progress bar
for text in tqdm(df['content'].astype(str), desc="Melabeli"):
    labels.append(tebak_sentimen(text))

# Simpan hasil ke kolom baru
df['label'] = labels

# Simpan ke CSV baru
df.to_csv(path_output, index=False)
print(f"\n🎉 SELESAI! Data berlabel disimpan di: {path_output}")
print(df['label'].value_counts())

Memulai pelabelan 1000 baris data INET...


Melabeli: 100%|██████████| 1000/1000 [32:59<00:00,  1.98s/it]


🎉 SELESAI! Data berlabel disimpan di: /content/drive/MyDrive/NLP_qwen/AADI_labeled.csv
label
Positif    517
Netral     337
Negatif    146
Name: count, dtype: int64


In [ ]:
path_input = "/content/drive/MyDrive/NLP_qwen/stockbit_BUMI_stream.csv"
path_output = "/content/drive/MyDrive/NLP_qwen/BUMI_labeled.csv"

df = pd.read_csv(path_input)

# Fungsi super pintar untuk memaksa Qwen menjawab 1 kata saja dengan Few-Shot Prompting
def tebak_sentimen(teks_komentar):
    # Prompt sistem dengan contoh agar Qwen langsung jago bahasa Stockbit
    messages = [
        {"role": "system", "content": """Kamu adalah ahli sentimen saham ritel Indonesia.
Tugasmu hanya menjawab dengan SATU KATA: 'Positif', 'Negatif', atau 'Netral'.
Contoh:
- 'Serok bawah mumpung diskon' -> Positif
- 'Aduh nyangkut di pucuk, CL aja deh' -> Negatif
- 'Hold dulu, nunggu laporan keuangan' -> Netral
- 'Bandar guyur terus' -> Negatif
- 'ARA besok nih!' -> Positif"""},
        {"role": "user", "content": f"Tentukan sentimen kalimat ini: '{teks_komentar}'"}
    ]

    text_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text_prompt], return_tensors="pt").to(model.device)

    # Generate jawaban maksimal 10 token biar cepat
    outputs = model.generate(**inputs, max_new_tokens=10, temperature=0.1, do_sample=False)

    # Ambil teks jawaban saja (buang prompt-nya)
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

    # Bersihkan jawaban (jaga-jaga kalau Qwen ngomong panjang lebar)
    hasil = response.strip().lower()
    if 'positif' in hasil: return 'Positif'
    elif 'negatif' in hasil: return 'Negatif'
    elif 'netral' in hasil: return 'Netral'
    else: return 'Netral' # Default kalau jawabannya aneh

print(f"Memulai pelabelan {len(df)} baris data INET...")
labels = []

# Proses satu per satu dengan progress bar
for text in tqdm(df['content'].astype(str), desc="Melabeli"):
    labels.append(tebak_sentimen(text))

# Simpan hasil ke kolom baru
df['label'] = labels

# Simpan ke CSV baru
df.to_csv(path_output, index=False)
print(f"\n🎉 SELESAI! Data berlabel disimpan di: {path_output}")
print(df['label'].value_counts())

Memulai pelabelan 1000 baris data INET...


Melabeli: 100%|██████████| 1000/1000 [27:48<00:00,  1.67s/it]


🎉 SELESAI! Data berlabel disimpan di: /content/drive/MyDrive/NLP_qwen/BUMI_labeled.csv
label
Positif    423
Negatif    315
Netral     262
Name: count, dtype: int64


In [5]:
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from tqdm import tqdm
from google.colab import drive
import re

# Hubungkan ke Drive
drive.mount('/content/drive')
path_input = "/content/drive/MyDrive/NLP_qwen/stockbit_ELSA_stream.csv"
path_output = "/content/drive/MyDrive/NLP_qwen/ELSA_labeled.csv"

df = pd.read_csv(path_input)

# Fungsi super pintar untuk memaksa Qwen menjawab 1 kata saja dengan Few-Shot Prompting
def tebak_sentimen(teks_komentar):
    # Prompt sistem dengan contoh agar Qwen langsung jago bahasa Stockbit
    messages = [
        {"role": "system", "content": """Kamu adalah ahli sentimen saham ritel Indonesia.
Tugasmu hanya menjawab dengan SATU KATA: 'Positif', 'Negatif', atau 'Netral'.
Contoh:
- 'Serok bawah mumpung diskon' -> Positif
- 'Aduh nyangkut di pucuk, CL aja deh' -> Negatif
- 'Hold dulu, nunggu laporan keuangan' -> Netral
- 'Bandar guyur terus' -> Negatif
- 'ARA besok nih!' -> Positif"""},
        {"role": "user", "content": f"Tentukan sentimen kalimat ini: '{teks_komentar}'"}
    ]

    text_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text_prompt], return_tensors="pt").to(model.device)

    # Generate jawaban maksimal 10 token biar cepat
    outputs = model.generate(**inputs, max_new_tokens=10, temperature=0.1, do_sample=False)

    # Ambil teks jawaban saja (buang prompt-nya)
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

    # Bersihkan jawaban (jaga-jaga kalau Qwen ngomong panjang lebar)
    hasil = response.strip().lower()
    if 'positif' in hasil: return 'Positif'
    elif 'negatif' in hasil: return 'Negatif'
    elif 'netral' in hasil: return 'Netral'
    else: return 'Netral' # Default kalau jawabannya aneh

print(f"Memulai pelabelan {len(df)} baris data INET...")
labels = []

# Proses satu per satu dengan progress bar
for text in tqdm(df['content'].astype(str), desc="Melabeli"):
    labels.append(tebak_sentimen(text))

# Simpan hasil ke kolom baru
df['label'] = labels

# Simpan ke CSV baru
df.to_csv(path_output, index=False)
print(f"\n🎉 SELESAI! Data berlabel disimpan di: {path_output}")
print(df['label'].value_counts())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Memulai pelabelan 1000 baris data INET...


Melabeli:  66%|██████▋   | 665/1000 [20:49<10:29,  1.88s/it]


OutOfMemoryError: CUDA out of memory. Tried to allocate 5.80 GiB. GPU 0 has a total capacity of 14.56 GiB of which 1.27 GiB is free. Including non-PyTorch memory, this process has 13.29 GiB memory in use. Of the allocated memory 11.80 GiB is allocated by PyTorch, and 1.36 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [6]:
# 1. Mengecek berapa banyak label yang selamat
jumlah_selamat = len(labels)
print(f"Data yang berhasil dilabeli: {jumlah_selamat} baris")

# 2. Gabungkan dengan data aslinya (potong sampai batas yang selamat)
df_terselamatkan = df.iloc[:jumlah_selamat].copy()
df_terselamatkan['label'] = labels

# 3. Simpan ke Drive!
path_part1 = "/content/drive/MyDrive/NLP_qwen/ELSA_part1.csv"
df_terselamatkan.to_csv(path_part1, index=False)
print(f"✅{jumlah_selamat} data aman di Drive!")

Data yang berhasil dilabeli: 665 baris
✅ Alhamdulillah, 665 data aman di Drive!


In [7]:
import gc
import torch

# Hapus cache Python
gc.collect()

# Kosongkan VRAM GPU secara paksa
torch.cuda.empty_cache()
print("🧹 VRAM GPU sudah dibersihkan dan siap kerja lagi!")

🧹 VRAM GPU sudah dibersihkan dan siap kerja lagi!


In [8]:
# Ambil sisa data yang belum diproses
df_sisa = df.iloc[jumlah_selamat:].copy()
print(f"🚀 Melanjutkan pelabelan untuk {len(df_sisa)} baris sisanya...")

labels_sisa = []

def tebak_sentimen_aman(teks_komentar):
    # Potong teks maksimal 500 karakter agar GPU tidak kaget
    teks_aman = teks_komentar[:500]

    messages = [
        {"role": "system", "content": "Kamu adalah ahli sentimen saham ritel Indonesia. Jawab SATU KATA: 'Positif', 'Negatif', atau 'Netral'."},
        {"role": "user", "content": f"Tentukan sentimen kalimat ini: '{teks_aman}'"}
    ]

    text_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text_prompt], return_tensors="pt").to(model.device)

    # Kita buang parameter temperature & do_sample=False agar warning hilang
    with torch.no_grad(): # Tambahan pengaman VRAM
        outputs = model.generate(**inputs, max_new_tokens=10, do_sample=False)

    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

    hasil = response.strip().lower()
    if 'positif' in hasil: return 'Positif'
    elif 'negatif' in hasil: return 'Negatif'
    elif 'netral' in hasil: return 'Netral'
    else: return 'Netral'

# Loop untuk sisa data
for text in tqdm(df_sisa['content'].astype(str), desc="Melabeli Sisa"):
    labels_sisa.append(tebak_sentimen_aman(text))

    # Bersihkan memori setiap 50 baris agar tidak numpuk
    if len(labels_sisa) % 50 == 0:
        torch.cuda.empty_cache()

# Gabungkan hasil sisa dan simpan
df_sisa['label'] = labels_sisa
path_part2 = "/content/drive/MyDrive/NLP_qwen/ELSA_part2.csv"
df_sisa.to_csv(path_part2, index=False)
print("\n🎉 SISA DATA SELESAI DAN TERSIMPAN!")

# Gabungkan part 1 dan part 2 menjadi INET yang utuh!
df_inet_final = pd.concat([df_terselamatkan, df_sisa], ignore_index=True)
df_inet_final.to_csv("/content/drive/MyDrive/NLP_qwen/ELSA_labeled_FINAL.csv", index=False)
print("🎊 FILE INET FINAL SUDAH JADI! Silakan cek Google Drive Anda.")

🚀 Melanjutkan pelabelan untuk 335 baris sisanya...


Melabeli Sisa: 100%|██████████| 335/335 [06:43<00:00,  1.20s/it]


🎉 SISA DATA SELESAI DAN TERSIMPAN!
🎊 FILE INET FINAL SUDAH JADI! Silakan cek Google Drive Anda.
